Model Notes:

Change in loss and hyper parameters:

replace softplus with tanh for gradient flow,
- Rank loss factor:
- triplet loss:
- alpha:

Model changes:

In [2]:
# ====================================================
# 🔧 STEP 1: Mount Google Drive
# ====================================================
from google.colab import drive
drive.mount('/content/drive')

# Create a working directory inside Drive
import os
WORK_DIR = '/content/drive/MyDrive/numeric_finetune_data'
os.makedirs(WORK_DIR, exist_ok=True)

Mounted at /content/drive


In [3]:
#!pip install -q transformers datasets peft accelerate wandb

In [4]:
"""
- embed and lineralayer weights included in lora
"""

'\n- embed and lineralayer weights included in lora\n'

In [5]:
# =========================================================
# ModernBERT + LoRA + Triplet Contrastive Training
# =========================================================

import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    DataCollatorWithPadding,
)

from peft import LoraConfig, get_peft_model
from accelerate import Accelerator
import wandb
from tqdm import tqdm

In [7]:
wandb.login()

wandb: WARNING Unable to verify login in offline mode.


False

In [8]:
!ls drive/MyDrive/numeric_finetune_data/NumerSense

data			      test_same_extracted.jsonl
happy-transformer	      test_same.jsonl
LICENSE			      test_sub_extracted.jsonl
Numeracy_600K_comment.json    test_sub.jsonl
README.md		      train_extracted_improved.jsonl
results			      train_extracted_improved_nodup.jsonl
src			      train_extracted.jsonl
test_extracted.jsonl	      train.jsonl
test_generic_extracted.jsonl  val_extracted.jsonl
test_generic.jsonl	      val.jsonl
test.jsonl


In [9]:

! head -2 drive/MyDrive/numeric_finetune_data/NumerSense/train_extracted.jsonl


{"anchor": "$27.9M City of Middletown, Connecticut Citigroup Global Markets Inc", "positive": "$29.61M City of Middletown, Connecticut Citigroup Global Markets Inc", "negative": "$23.6M City of Middletown, Connecticut Citigroup Global Markets Inc", "number": "27.9", "positive_rewritten": "Citigroup Global Markets Inc in Middletown, Connecticut with a value of $29.61M", "negative_rewritten": "Citigroup Global Markets Inc in Middletown, Connecticut with a value of $23.6M", "positive_number": "29.61", "negative_number": 23.6}
{"anchor": "Ex-N.Y. Senate leader Bruno asks state for $2.4 million in legal fees", "positive": "Ex-N.Y. Senate leader Bruno asks state for $2.07 million in legal fees", "negative": "Ex-N.Y. Senate leader Bruno asks state for $3.36 million in legal fees", "number": "2.4", "positive_rewritten": "Former New York Senate leader Bruno requests $2.07 million from the state for legal expenses", "negative_rewritten": "Former New York Senate leader Bruno seeks $3.36 million f

In [10]:
# =========================================================
# CONFIG
# =========================================================

MODEL_NAME = "answerdotai/ModernBERT-base"

LORA_MODEL = "drive/MyDrive/numeric_finetune_data/trained_model/modernbert_lora_contrastive_CDL_GPTl_hyper"

TRAIN_FILE = "drive/MyDrive/numeric_finetune_data/NumerSense/train_extracted_improved_nodup.jsonl"     # ~79k
VAL_FILE   = "drive/MyDrive/numeric_finetune_data/NumerSense/val_extracted.jsonl"       # ~10k

MAX_LENGTH = 128
BATCH_SIZE = 32
EPOCHS = 3
LR = 2e-5
WEIGHT_DECAY = 0.01
MARGIN = 0.2

WANDB_PROJECT = "modernbert-numeracy-lora-corrected-final"

# Checkpointing settings
CHECKPOINT_DIR = os.path.join(WORK_DIR, "checkpoints_final_margin")
RESUME_FROM_CHECKPOINT = False # Set to True to resume training
SAVE_CHECKPOINT_STEPS = 500 # Save a checkpoint every N steps

In [11]:
# =========================================================
# DATASET
# =========================================================

class TripletDataset(Dataset):
    def __init__(self, path, tokenizer):
        self.data = []
        with open(path) as f:
            for line in f:
                self.data.append(json.loads(line))
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "anchor": self.tokenizer(
                item["anchor"],
                truncation=True,
                max_length=MAX_LENGTH
            ),
            "positive": self.tokenizer(
                item["positive_rewritten"],
                truncation=True,
                max_length=MAX_LENGTH
            ),
            "negative": self.tokenizer(
                item["negative_rewritten"],
                truncation=True,
                max_length=MAX_LENGTH
            ),
            "positive_number": float(item["positive_number"]),
            "negative_number": float(item["negative_number"]),
            "anchor_number": float(item["number"]),
        }

In [12]:
# =========================================================
# COLLATOR (DataCollatorWithPadding for triplets)
# =========================================================

def make_triplet_collator(tokenizer):
    base_collator = DataCollatorWithPadding(tokenizer)

    def collate(batch):
        return {
            "anchor": base_collator([b["anchor"] for b in batch]),
            "positive": base_collator([b["positive"] for b in batch]),
            "negative": base_collator([b["negative"] for b in batch]),
            "positive_number": torch.tensor([b["positive_number"] for b in batch], dtype=torch.float),
            "negative_number": torch.tensor([b["negative_number"] for b in batch], dtype=torch.float),
            "anchor_number": torch.tensor([b["anchor_number"] for b in batch], dtype=torch.float),
        }

    return collate

In [13]:
# =========================================================
# MEAN POOLING (IMPORTANT FOR MODERNBERT)
# =========================================================

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).float()
    return (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)

In [14]:
class NumericHead(nn.Module):
    def __init__(self, emb_dim: int, latent_dim: int = 12, dropout: float = 0.1):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(emb_dim, latent_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(latent_dim * 2, latent_dim),
            nn.GELU(),
        )
        self.magnitude_head = nn.Linear(latent_dim, 1)
        self.rank_head      = nn.Linear(latent_dim, 1)

        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.5)
                nn.init.zeros_(m.bias)

    def forward(self, emb: torch.Tensor):
        latent = self.trunk(emb)
        return self.magnitude_head(latent).squeeze(-1), self.rank_head(latent).squeeze(-1)


class ContrastiveModel(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        hidden_size = encoder.config.hidden_size

        # replaces nn.Linear(hidden_size, 1)
        # raw emb → 12-dim latent → two scalar outputs
        self.numeric_head = NumericHead(hidden_size, latent_dim=12, dropout=0.1)

        # unchanged — clean directional space for cosine losses
        self.metric_proj = nn.Linear(hidden_size, hidden_size)

    def encode(self, batch_part):
        out = self.encoder(
            input_ids=batch_part["input_ids"],
            attention_mask=batch_part["attention_mask"],
        )
        return mean_pooling(out, batch_part["attention_mask"])  # raw

    def forward(self, batch):
        a_emb = self.encode(batch["anchor"])
        p_emb = self.encode(batch["positive"])
        n_emb = self.encode(batch["negative"])

        # numeric_head now returns (magnitude_score, rank_score) for each
        a_mag, a_rnk = self.numeric_head(a_emb)
        p_mag, p_rnk = self.numeric_head(p_emb)
        n_mag, n_rnk = self.numeric_head(n_emb)

        # metric proj unchanged
        a_proj = F.normalize(self.metric_proj(a_emb), dim=-1)
        p_proj = F.normalize(self.metric_proj(p_emb), dim=-1)
        n_proj = F.normalize(self.metric_proj(n_emb), dim=-1)

        return {
            "a_emb":   a_proj,  # triplet + log-distance loss
            "p_emb":   p_proj,
            "n_emb":   n_proj,
            "a_mag":   a_mag,   # head_loss  (predicts log1p(value))
            "p_mag":   p_mag,
            "n_mag":   n_mag,
            "a_score": a_rnk,   # rank_loss  (pairwise ordering)
            "p_score": p_rnk,   # kept as a_score/p_score/n_score so your
            "n_score": n_rnk,   # loss function call signature doesn't change
        }

In [15]:
import torch
import torch.nn.functional as F

def improved_numeric_loss(
    anchor_emb, pos_emb, neg_emb,
    anchor_mag, pos_mag, neg_mag,
    anchor_rnk, pos_rnk, neg_rnk,
    anchor_value, pos_value, neg_value,
    base_margin=0.2,
    alpha=0.4,
    beta=0.7,
    eps=1e-8,
    sign_temp=0.5,
    min_rank_gap=0.05,
):
    # --------------------------------------------------
    # 1. Normalize embeddings (metric space only)
    # --------------------------------------------------
    anchor = F.normalize(anchor_emb, dim=-1)
    pos    = F.normalize(pos_emb,    dim=-1)
    neg    = F.normalize(neg_emb,    dim=-1)

    # --------------------------------------------------
    # 2. Cosine distances ∈ [0, 2]
    # --------------------------------------------------
    pos_cos = 1.0 - F.cosine_similarity(anchor, pos, dim=-1)
    neg_cos = 1.0 - F.cosine_similarity(anchor, neg, dim=-1)
    pn_cos  = 1.0 - F.cosine_similarity(pos,    neg, dim=-1)

    # --------------------------------------------------
    # 3. Log-space numeric distances
    # --------------------------------------------------
    log_a = torch.log1p(anchor_value + eps)
    log_p = torch.log1p(pos_value    + eps)
    log_n = torch.log1p(neg_value    + eps)

    log_pos = torch.abs(log_a - log_p)
    log_neg = torch.abs(log_a - log_n)
    log_pn  = torch.abs(log_p - log_n)

    # --------------------------------------------------
    # 4. Dynamic triplet loss
    #    margin grows when anchor-neg gap is larger than anchor-pos gap
    # --------------------------------------------------
    dyn_margin   = base_margin * (1.0 + (log_neg - log_pos).clamp(min=0))
    triplet_loss = F.relu(pos_cos - neg_cos + dyn_margin).mean()

    # --------------------------------------------------
    # 5. Log-distance alignment
    #    scaling(x) = x / (1 + x) maps [0, ∞) → [0, 1)
    #    forces cosine distances to be proportional to log-value distances
    # --------------------------------------------------
    def scale(x):
        return x / (1.0 + x)

    log_distance_loss = (
        F.mse_loss(pos_cos / 2.0, scale(log_pos)) +
        F.mse_loss(neg_cos / 2.0, scale(log_neg)) +
        F.mse_loss(pn_cos  / 2.0, scale(log_pn))
    ) / 3.0

    # --------------------------------------------------
    # 6. Head loss
    #    magnitude_head is trained to predict log1p(value) directly.
    #    uses anchor_mag / pos_mag / neg_mag — the dedicated magnitude
    #    output from NumericHead, not the rank output.
    # --------------------------------------------------
    head_loss = (
        F.mse_loss(anchor_mag, log_a) +
        F.mse_loss(pos_mag,    log_p) +
        F.mse_loss(neg_mag,    log_n)
    ) / 3.0

    # --------------------------------------------------
    # 7. Rank loss  (Fix 1 + Fix 2)
    #
    #    uses anchor_rnk / pos_rnk / neg_rnk — the dedicated rank
    #    output from NumericHead. gradients no longer conflict with
    #    head_loss because the two outputs have separate weights.
    #
    #    Fix 1 — tanh replaces torch.sign
    #      sign(x) has zero gradient everywhere; tanh(x / temp) is
    #      smooth and differentiable. temp controls sharpness:
    #        temp → 0  :  approaches hard sign
    #        temp = 0.5:  soft but directional  ← default
    #        temp → ∞  :  collapses toward zero, loses direction signal
    #
    #    Fix 2 — margin scaled by value gap
    #      pairs with nearly identical values get a near-zero required
    #      margin. stops easy near-tie pairs from dominating the gradient.
    #
    #    masking — pairs below min_rank_gap are excluded entirely.
    #      prevents genuinely ambiguous pairs from sending contradictory
    #      gradients when values are too close to have a meaningful order.
    # --------------------------------------------------
    diff_ap = log_a - log_p
    diff_an = log_a - log_n

    # smooth differentiable direction signal
    ap_sign = torch.tanh(diff_ap / sign_temp)
    an_sign = torch.tanh(diff_an / sign_temp)

    # how far apart the values are, clamped to [0, 1]
    gap_ap = torch.abs(diff_ap).clamp(max=1.0)
    gap_an = torch.abs(diff_an).clamp(max=1.0)

    # exclude near-tie pairs
    mask_ap = (torch.abs(diff_ap) > min_rank_gap).float()
    mask_an = (torch.abs(diff_an) > min_rank_gap).float()

    # pairwise score differences from the rank head
    ap = anchor_rnk - pos_rnk
    an = anchor_rnk - neg_rnk

    # softplus(gap * margin - sign * score_diff):
    #   → zero when score_diff has right sign and exceeds gap-scaled margin
    #   → positive and growing when direction is wrong or gap is unsatisfied
    loss_ap = F.softplus(gap_ap * base_margin - ap_sign * ap)
    loss_an = F.softplus(gap_an * base_margin - an_sign * an)

    rank_loss = (
        (mask_ap * loss_ap).sum() / (mask_ap.sum() + eps) +
        (mask_an * loss_an).sum() / (mask_an.sum() + eps)
    ) / 2.0

    # --------------------------------------------------
    # 8. Weighted grouping
    #
    #    metric_loss      = triplet + log_distance
    #                       pure embedding space, no head involved
    #
    #    supervision_loss = β · head  +  (1−β) · rank
    #                       β=0.7 → head gets more gradient, right call
    #                       given head was overfitting and rank was flat
    #
    #    total_loss       = α · metric  +  (1−α) · supervision
    #                       α=0.4 → slight supervision emphasis while
    #                       metric space continues to improve
    # --------------------------------------------------
    metric_loss      = triplet_loss + log_distance_loss
    supervision_loss = beta * head_loss + (1 - beta) * rank_loss
    total_loss       = alpha * metric_loss + (1 - alpha) * supervision_loss

    components = {
        "triplet":     triplet_loss.item(),
        "log_dist":    log_distance_loss.item(),
        "head":        head_loss.item(),
        "rank":        rank_loss.item(),
        "metric":      metric_loss.item(),
        "supervision": supervision_loss.item(),
        "total":       total_loss.item(),
    }
    return total_loss, components

In [14]:
del base_model
del model


NameError: name 'base_model' is not defined

In [15]:
#base_model = AutoModel.from_pretrained(MODEL_NAME)
#del base_model


In [16]:
layers_names = []
for name, module in base_model.named_modules():
    #print(name, len(name.split('.')))
    if len(name.split('.')) <= 2:
        continue
    layers_names.append(name.split('.')[-1])
set(layers_names)

NameError: name 'base_model' is not defined

In [ ]:
for name, module in base_model.named_modules():
    #print(name, len(name.split('.')))

    layers_names.append(name.split('.')[-1])
set(layers_names)

NameError: name 'base_model' is not defined

In [16]:
accelerator = Accelerator()
#wandb.init(project=WANDB_PROJECT)

# Tokenizer & Base Model

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModel.from_pretrained(MODEL_NAME)



# -----------------------------------------------------
# LoRA CONFIG (attention layers only)
# -----------------------------------------------------
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,

    target_modules=[
        "Wqkv",
        "out_proj",
        "Wi",
        "Wo",
        "tok_embeddings" # ModernBERT uses this for word embeddings
    ],
    bias="none",
    task_type="FEATURE_EXTRACTION",
    # CRITICAL: Directly train these modules in full precision.
    # This avoids the "Identity" error and allows normalization to adapt.
    modules_to_save=[
        "numeric_head",
        "metric_proj",
        "attn_norm",   # ModernBERT normalization
        "mlp_norm",    # ModernBERT normalization
        "emb_norm"     # Post-embedding normalization
    ],
)

base_model = get_peft_model(base_model, lora_config)
model = ContrastiveModel(base_model)
"""
# LORA loading
# Load base architecture
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModel.from_pretrained(MODEL_NAME)

# Attach LoRA weights
base_model = PeftModel.from_pretrained(
    base_model,
    LORA_MODEL,
    is_trainable=True
)

model = ContrastiveModel(base_model)
"""

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


'\n# LORA loading\n# Load base architecture\nfrom peft import PeftModel\n\ntokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)\nbase_model = AutoModel.from_pretrained(MODEL_NAME)\n\n# Attach LoRA weights\nbase_model = PeftModel.from_pretrained(\n    base_model,\n    LORA_MODEL,\n    is_trainable=True\n)\n\nmodel = ContrastiveModel(base_model)\n'

In [17]:
#model.encoder.print_trainable_parameters()


In [18]:
# -----------------------------------------------------
# Data
# -----------------------------------------------------
train_dataset = TripletDataset(TRAIN_FILE, tokenizer)
val_dataset   = TripletDataset(VAL_FILE, tokenizer)

collate_fn = make_triplet_collator(tokenizer)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

In [19]:
"""
from torch.utils.data import Subset
# Take only the first 100 samples
train_subset_dataset = Subset(train_dataset, range(200))
val_subset_dataset = Subset(train_dataset, range(200))
train_loader = DataLoader(train_subset_dataset, batch_size=32)
train_loader = DataLoader(
    train_subset_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_subset_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)
"""

'\nfrom torch.utils.data import Subset\n# Take only the first 100 samples\ntrain_subset_dataset = Subset(train_dataset, range(200))\nval_subset_dataset = Subset(train_dataset, range(200))\ntrain_loader = DataLoader(train_subset_dataset, batch_size=32)\ntrain_loader = DataLoader(\n    train_subset_dataset,\n    batch_size=BATCH_SIZE,\n    shuffle=True,\n    collate_fn=collate_fn\n)\n\nval_loader = DataLoader(\n    val_subset_dataset,\n    batch_size=BATCH_SIZE,\n    shuffle=False,\n    collate_fn=collate_fn\n)\n'

In [20]:
counter = 0
for i, item in enumerate(train_dataset.data):
    a = float(item["number"])
    p = float(item["positive_number"])
    n = float(item["negative_number"])

    if a <= 0 or p <= 0 or n <= 0:
        counter +=1
        #print(f"idx={i}  anchor={a}  pos={p}  neg={n}")
print(counter)

4444


In [21]:
batch = next(iter(train_loader))
batch

{'anchor': {'input_ids': tensor([[50281,    38,    42,  ..., 50283, 50283, 50283],
         [50281,    35,  2354,  ..., 50283, 50283, 50283],
         [50281, 46009,  1277,  ..., 50283, 50283, 50283],
         ...,
         [50281, 38299,  5472,  ..., 50283, 50283, 50283],
         [50281,  5648,    41,  ..., 50283, 50283, 50283],
         [50281, 34429,  1277,  ..., 50283, 50283, 50283]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         ...,
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0]])},
 'positive': {'input_ids': tensor([[50281,    38,    42,  ..., 50283, 50283, 50283],
         [50281,    35,  2354,  ..., 50283, 50283, 50283],
         [50281,    53,  9033,  ..., 50283, 50283, 50283],
         ...,
         [50281, 38299,  5472,  ..., 50283, 50283, 50283],
         [50281, 12677,    42,  ..., 50283, 50283, 50283],
         [50281, 34429,  12

In [22]:
RESUME_FROM_CHECKPOINT = True
RESUME_FROM_CHECKPOINT

True

In [23]:
# Split parameters into three groups
lora_params = [
    p for n, p in model.named_parameters()
    if "lora_" in n and p.requires_grad
]
head_params = list(model.numeric_head.parameters())

# If using metric_proj:
proj_params = list(model.metric_proj.parameters())

optimizer = torch.optim.AdamW([
    {"params": lora_params,  "lr": 1e-4,  "weight_decay": 0.01},
    {"params": head_params,  "lr": 1e-3,  "weight_decay": 0.01},
   {"params": proj_params, "lr": 5e-3,  "weight_decay": 0.01},
], betas=(0.9, 0.999), eps=1e-8)

In [24]:
#proj_params

In [25]:


model, optimizer, train_loader, val_loader = accelerator.prepare(
    model, optimizer, train_loader, val_loader
)

# =========================================================
# CHECKPOINTING: Resume from Checkpoint (moved here after prepare)
# =========================================================
start_epoch = 0
global_step = 0

# Ensure the checkpoint directory exists
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

if RESUME_FROM_CHECKPOINT:
    # Check for a specific file to confirm checkpoint existence
    # Accelerator saves a 'pytorch_model.bin' and 'optimizer.bin' along with other states.
    if os.path.exists(os.path.join(CHECKPOINT_DIR, "model.safetensors")):
        accelerator.load_state(CHECKPOINT_DIR)
        accelerator.print(f"Resuming training from checkpoint in {CHECKPOINT_DIR}")

        # Load metadata (epoch and global_step) if available
        metadata_path = os.path.join(CHECKPOINT_DIR, "training_metadata.json")
        if accelerator.is_main_process and os.path.exists(metadata_path):
            with open(metadata_path, 'r') as f:
                metadata = json.load(f)
                start_epoch = metadata.get("epoch", 0)
                global_step = metadata.get("global_step", 0)
            accelerator.print(f"Resumed epoch: {start_epoch}, global_step: {global_step}")
        elif not accelerator.is_main_process:
            # All processes need to wait for the main process to load metadata
            accelerator.wait_for_everyone()
            if os.path.exists(metadata_path):
                 with open(metadata_path, 'r') as f:
                    metadata = json.load(f)
                    start_epoch = metadata.get("epoch", 0)
                    global_step = metadata.get("global_step", 0)
    else:
        accelerator.print(f"No checkpoint found at {CHECKPOINT_DIR}. Starting fresh.")
else:
    accelerator.print("Starting training from scratch (RESUME_FROM_CHECKPOINT is False).")

accelerator.print(f"Initial epoch: {start_epoch}, initial global_step: {global_step}")

Resuming training from checkpoint in /content/drive/MyDrive/numeric_finetune_data/checkpoints_final_margin
Resumed epoch: 4, global_step: 6000
Initial epoch: 4, initial global_step: 6000


In [26]:
#start_epoch= 3
start_epoch,global_step,list(range(start_epoch, EPOCHS+4))

(4, 6000, [4, 5, 6])

In [27]:
wandb.init(project=WANDB_PROJECT) # Initialize wandb with project name

# Ensure the checkpoint directory exists
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

for epoch in range(start_epoch, EPOCHS+4): # Start from 'start_epoch'
    model.train()
    total_loss = 0.0
    triplet_loss = 0.0
    log_dist = 0.0
    head = 0.0
    rank = 0.0

    progress_bar = tqdm(
        train_loader,
        disable=not accelerator.is_main_process,
        desc=f"Epoch {epoch+1}"
    )

    for step, batch in enumerate(progress_bar):

        # Calculate current_global_step, accounting for resumed training
        current_global_step = global_step + (epoch - start_epoch) * len(train_loader) + step
        #print(batch)
        # if epoch == 5 and step < 17500:
        #     continue
        out = model(batch)
        loss, components = improved_numeric_loss(
            anchor_emb=out["a_emb"], pos_emb=out["p_emb"], neg_emb=out["n_emb"],
            anchor_mag=out["a_mag"], pos_mag=out["p_mag"], neg_mag=out["n_mag"],
            anchor_rnk=out["a_score"], pos_rnk=out["p_score"], neg_rnk=out["n_score"],
            anchor_value=batch["anchor_number"],
            pos_value=batch["positive_number"],
            neg_value=batch["negative_number"],
            alpha=0.4, beta=0.7, sign_temp=0.5,
        )

        accelerator.backward(loss)
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        triplet_loss += components["triplet"]
        log_dist += components["log_dist"]
        head += components["head"]
        rank += components["rank"]

        # Log every 100 steps for WandB or if it's the very first step
        if (current_global_step + 1) % 100 == 0 or current_global_step == 0:
            accelerator.print(f"Step {current_global_step+1} | Loss {loss.item():.4f}")
            accelerator.print(f"Triplet {components['triplet']} | log_dist {components['log_dist']}")
            accelerator.print(f" head {components['head']} | rank {components['rank']}")
            wandb.log({
                "train_loss_step": loss.item(),
                "global_step": current_global_step + 1,
                "epoch": epoch
            })

        # Checkpoint saving logic every SAVE_CHECKPOINT_STEPS
        if (current_global_step + 1) % SAVE_CHECKPOINT_STEPS == 0:
            accelerator.save_state(CHECKPOINT_DIR)
            # Save metadata (epoch, global_step) alongside the model
            if accelerator.is_main_process:
                metadata = {"epoch": epoch, "global_step": current_global_step + 1}
                with open(os.path.join(CHECKPOINT_DIR, "training_metadata.json"), 'w') as f:
                    json.dump(metadata, f)
            accelerator.wait_for_everyone() # Ensure all processes save before proceeding
            accelerator.print(f"Checkpoint saved at global step {current_global_step+1} to {CHECKPOINT_DIR}")


    train_loss = total_loss / len(train_loader)
    triplet_loss /= len(train_loader)
    log_dist /= len(train_loader)
    head /= len(train_loader)
    rank /= len(train_loader)


    # -------------------------------------------------
    # VALIDATION
    # -------------------------------------------------
    model.eval()
    val_loss = 0.0
    val_triplet_loss = 0.0
    val_log_distance = 0.0
    val_head_loss = 0.0
    val_rank_loss = 0.0


    with torch.no_grad():
        for batch in val_loader:
            out = model(batch)
            loss, components = improved_numeric_loss(
                                    anchor_emb=out["a_emb"], pos_emb=out["p_emb"], neg_emb=out["n_emb"],
                                    anchor_mag=out["a_mag"], pos_mag=out["p_mag"], neg_mag=out["n_mag"],
                                    anchor_rnk=out["a_score"], pos_rnk=out["p_score"], neg_rnk=out["n_score"],
                                    anchor_value=batch["anchor_number"],
                                    pos_value=batch["positive_number"],
                                    neg_value=batch["negative_number"],
                                    alpha=0.4, beta=0.7, sign_temp=0.5,
                                )
            val_loss += loss.item()
            val_triplet_loss += components["triplet"]
            val_log_distance += components["log_dist"]
            val_head_loss += components["head"]
            val_rank_loss += components["rank"]

    val_loss /= len(val_loader)
    val_triplet_loss /= len(val_loader)
    val_log_distance /= len(val_loader)
    val_head_loss /= len(val_loader)
    val_rank_loss /= len(val_loader)

    wandb.log({
         "epoch": epoch + 1,
         "train_loss": train_loss,
         "val_loss": val_loss
     })

    accelerator.print(
        f"Epoch {epoch+1} | Train: {train_loss:.4f} | Val: {val_loss:.4f}"

    )
    accelerator.print(f"Train: Triplet {triplet_loss} | log_dist {log_dist}")
    accelerator.print(f"Valid: Triplet {val_triplet_loss} | log_dist {val_log_distance}")
    accelerator.print(f"Train: head {head} | rank {rank}")
    accelerator.print(f"Valid: head {val_head_loss} | rank {val_rank_loss}")

    # -----------------------------------------------------
    # SAVE MODEL AND TOKENIZER AFTER EACH EPOCH (for final model export)
    # -----------------------------------------------------
    accelerator.wait_for_everyone()
    unwrapped = accelerator.unwrap_model(model)
    save_path = os.path.join(WORK_DIR, f"trained_model/modernbert_lora_contrastive_corrected_CDL_GPTl_hyper_epoch_{epoch+1}")
    os.makedirs(save_path, exist_ok=True)
    unwrapped.encoder.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    accelerator.print(f"Model and tokenizer saved for epoch {epoch+1} to {save_path}")

    if accelerator.is_main_process:
        metadata = {"epoch": epoch, "global_step": current_global_step + 1}
        with open(os.path.join(CHECKPOINT_DIR, "training_metadata.json"), 'w') as f:
            json.dump(metadata, f)
    accelerator.wait_for_everyone() # Ensure all processes save before proceeding
    accelerator.print(f"Checkpoint saved at global step {current_global_step+1} to {CHECKPOINT_DIR}")

# -----------------------------------------------------
# WANDB FINISH (after all epochs complete)
# -----------------------------------------------------
wandb.finish()

Epoch 5:   4%|▍         | 100/2596 [02:54<58:27,  1.41s/it] 

Step 6100 | Loss 0.1240
Triplet 0.13279105722904205 | log_dist 0.00554971769452095
 head 0.04101550579071045 | rank 0.2857242822647095


Epoch 5:   8%|▊         | 200/2596 [05:09<54:27,  1.36s/it]

Step 6200 | Loss 0.1389
Triplet 0.18091747164726257 | log_dist 0.0084771066904068
 head 0.034289222210645676 | rank 0.2705400586128235


Epoch 5:  12%|█▏        | 300/2596 [07:23<1:00:27,  1.58s/it]

...

Epoch 7:  97%|█████████▋| 2508/2596 [57:04<02:01,  1.38s/it]

Step 13700 | Loss 0.1456
Triplet 0.24856078624725342 | log_dist 0.008090592920780182
 head 0.028359755873680115 | rank 0.1721065193414688


Epoch 7: 100%|██████████| 2596/2596 [59:02<00:00,  1.36s/it]


Epoch 7 | Train: 0.1267 | Val: 0.1741
Train: Triplet 0.15094980653461487 | log_dist 0.009094270569102291
Valid: Triplet 0.09040265939890957 | log_dist 0.0075992980571941305
Train: head 0.05537762944080733 | rank 0.21929017812564672
Valid: head 0.20793851524686965 | rank 0.26443999222455883
Model and tokenizer saved for epoch 7 to /content/drive/MyDrive/numeric_finetune_data/trained_model/modernbert_lora_contrastive_corrected_CDL_GPTl_hyper_epoch_7
Checkpoint saved at global step 13788 to /content/drive/MyDrive/numeric_finetune_data/checkpoints_final_margin


epoch,▁▁▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▆▆▆▆▆▆▆▆▆▆▆▆█
global_step,▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇███
train_loss,█▇▁
train_loss_step,▄▄▃▃▃▇▄▃▄▃▃▆▅▄▂▄▅▄▄▃▄▇▄▄▃▄▃▅▃▅▂█▂▃▃▅▁▇▄▅
val_loss,▁█▆
epoch,7
global_step,13700
train_loss,0.12675
train_loss_step,0.14555
val_loss,0.17413


In [29]:
from google.colab import runtime
runtime.unassign()

In [28]:
# for name, param in model.named_parameters():
#     if param.requires_grad:
#         print(name)


In [ ]:
#model

In [ ]:
!ls

sample_data


In [ ]:
accelerator.wait_for_everyone()
unwrapped = accelerator.unwrap_model(model)
unwrapped.encoder.save_pretrained("modernbert_lora_contrastive_corrected_dynamic")
tokenizer.save_pretrained("modernbert_lora_contrastive_corrected_dynamic")

wandb.finish()

In [ ]:
epoch=0
accelerator.wait_for_everyone()
unwrapped = accelerator.unwrap_model(model)
save_path = os.path.join(WORK_DIR, f"trained_model/modernbert_lora_contrastive_corrected_ML_epoch_{epoch+1}")
os.makedirs(save_path, exist_ok=True)
unwrapped.encoder.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
accelerator.print(f"Model and tokenizer saved for epoch {epoch+1} to {save_path}")

Model and tokenizer saved for epoch 1 to /content/drive/MyDrive/numeric_finetune_data/trained_model/modernbert_lora_contrastive_corrected_ML_epoch_1


In [ ]:
#!mkdir -p drive/MyDrive/numeric_finetune_data/trained_model


In [ ]:
! cp -r modernbert_lora_contrastive-corrected2 drive/MyDrive/numeric_finetune_data/trained_model/